# Directional lower/upper masks for Rashomon bounds

This small example grows a certified parameter box only in the direction of a proposed policy update. For each parameter entry:

- a negative proposal leaves the lower bound free and freezes the upper bound;
- a positive proposal freezes the lower bound and leaves the upper bound free;
- a zero proposal freezes both bounds.

In these APIs, `True` means **frozen at the nominal parameter value**. The masks constrain the shape of the box; the usual certificate still checks the complete resulting box.

In [ ]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "core").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
CORE_PATH = PROJECT_ROOT / "core"
if str(CORE_PATH) not in sys.path:
    sys.path.insert(0, str(CORE_PATH))

from src.interval_utils import compute_rashomon_set

torch.manual_seed(0)


## 1. A tiny safe classification problem

The two-output linear model strongly prefers output 0. The zero inputs make this example especially easy to inspect: weight changes do not affect its predictions, while the bias intervals are still checked by the certificate.

In [ ]:
model = torch.nn.Sequential(torch.nn.Linear(2, 2))
with torch.no_grad():
    model[0].weight.zero_()
    model[0].bias.copy_(torch.tensor([2.0, 0.0]))

inputs = torch.zeros(8, 2)
targets = torch.tensor([[1.0, 0.0]]).expand(len(inputs), -1).clone()
dataset = torch.utils.data.TensorDataset(inputs, targets)
nominal_parameters = [parameter.detach().clone() for parameter in model.parameters()]

print("Nominal output:", model(inputs[:1]).detach().squeeze().tolist())


## 2. Convert the proposed update into side-specific masks

The proposed changes must follow the same order and shapes as `model.parameters()`. The formulas below implement the sign table directly. A small tolerance can treat negligible proposals as zero.

In [ ]:
proposed_changes = [
    torch.tensor([[-0.10, 0.08], [0.00, -0.03]]),  # weight
    torch.tensor([-0.05, 0.04]),                    # bias
]

tolerance = 1e-12
param_l_mask = [change >= -tolerance for change in proposed_changes]
param_u_mask = [change <= tolerance for change in proposed_changes]

for index, (change, lower_mask, upper_mask) in enumerate(
    zip(proposed_changes, param_l_mask, param_u_mask)
):
    print(f"parameter tensor {index}")
    print("  proposed change:\n", change)
    print("  freeze lower:\n", lower_mask)
    print("  freeze upper:\n", upper_mask)


## 3. Weight growth by the proposed change magnitude

The masks decide which sides are allowed to move. This custom objective decides how strongly to reward each allowed side, using the absolute proposed change as its weight.

In [ ]:
def directional_objective(bounded_model, _alpha):
    utility = torch.zeros((), device=bounded_model.device)
    total_weight = torch.zeros((), device=bounded_model.device)

    for p_l, p_n, p_u, change in zip(
        bounded_model.param_l,
        bounded_model.param_n,
        bounded_model.param_u,
        proposed_changes,
    ):
        change = change.to(p_l.device)
        negative_weight = (-change).clamp_min(0)
        positive_weight = change.clamp_min(0)
        negative_radius = p_n - p_l
        positive_radius = p_u - p_n

        utility = utility + (negative_weight * negative_radius).sum()
        utility = utility + (positive_weight * positive_radius).sum()
        total_weight = total_weight + negative_weight.sum() + positive_weight.sum()

    return utility / total_weight.clamp_min(1e-12)


In [ ]:
result = compute_rashomon_set(
    model,
    dataset,
    accuracy=1.0,
    batch_size=len(dataset),
    certificate_samples=len(dataset),
    n_iters=25,
    primal_learning_rate=0.3,
    use_schedule=False,
    temperatures={None: 0.1},
    custom_objective=directional_objective,
    param_l_mask=param_l_mask,
    param_u_mask=param_u_mask,
)

bounded_model = result.bounded_models[-1]
print("Certificate:", result.certificates[-1][0])


## 4. Inspect and verify the asymmetric intervals

The assertions check the key guarantee: every masked side is exactly equal to the nominal parameter. The printed rows show the resulting interval for each scalar parameter.

In [ ]:
rows = []
for tensor_index, (p_l, p_n, p_u, change, lower_mask, upper_mask) in enumerate(
    zip(
        bounded_model.param_l,
        bounded_model.param_n,
        bounded_model.param_u,
        proposed_changes,
        param_l_mask,
        param_u_mask,
    )
):
    torch.testing.assert_close(p_l[lower_mask], p_n[lower_mask], rtol=0, atol=0)
    torch.testing.assert_close(p_u[upper_mask], p_n[upper_mask], rtol=0, atol=0)

    for entry_index in range(change.numel()):
        rows.append(
            (
                tensor_index,
                entry_index,
                change.flatten()[entry_index].item(),
                p_l.flatten()[entry_index].item(),
                p_n.flatten()[entry_index].item(),
                p_u.flatten()[entry_index].item(),
            )
        )

print("tensor entry   change       lower     nominal       upper")
for tensor_index, entry_index, change, lower, nominal, upper in rows:
    print(
        f"{tensor_index:>6} {entry_index:>5} "
        f"{change:>9.3f} {lower:>11.4f} {nominal:>11.4f} {upper:>11.4f}"
    )

print("\nAll frozen sides remained exactly at their nominal values.")


## Key points

- `param_l_mask[i][j] = True` freezes the lower bound for parameter entry `j`.
- `param_u_mask[i][j] = True` freezes the upper bound.
- The older `param_mask` remains available and freezes both sides. If it is supplied together with the new masks, the masks are combined with boolean OR.
- Gradient hooks stop the optimiser updating a frozen entry, and projection restores it to the nominal value after every step.
- Side-specific masks control direction; the custom objective controls how growth is allocated among the allowed directions.